# Notebook 06: Data Augmentation for NLP

## 📚 Historical Context

**Timeline**: Evolution of NLP Data Augmentation (2011-2024)

**Computer Vision Era (2011-2015)**:
- **2012**: AlexNet wins ImageNet with aggressive augmentation
- **Techniques**: Rotation, flip, crop, color jitter
- **Success**: 10-30% accuracy improvement
- **NLP**: Left behind - text is discrete, not continuous!

**Early NLP Augmentation (2014-2017)**:
- **2014**: Synonym replacement (WordNet)
- **2015**: Back-translation for machine translation
- **2016**: Contextual word replacement
- **Challenge**: Hard to preserve semantics

**Deep Learning Era (2018-2021)**:
- **2018**: EDA (Easy Data Augmentation) paper
  - Simple rules: synonym, insert, swap, delete
  - +0.8% to +3.0% accuracy on benchmark tasks
- **2019**: Back-translation with BERT/GPT
- **2019**: Conditional BERT for contextual augmentation
- **2020**: Adversarial training (add perturbations)

**LLM Era (2022-2024)**:
- **2022**: GPT-3 for paraphrasing and generation
- **2023**: Instruction-based augmentation
- **2023**: Few-shot generation with Llama/GPT-4
- **Current**: Quality > quantity (careful filtering)

**Key Insight**:
- **Computer Vision**: Augmentation is standard practice
- **NLP**: More challenging, but can help (especially small datasets)
- **Modern approach**: LLM-based paraphrasing with quality filters

## 🎯 What You'll Learn

1. **Back-Translation** - Translate and translate back
2. **Synonym Replacement** - WordNet-based substitution
3. **Contextual Replacement** - BERT masked prediction
4. **EDA Techniques** - Random insert, delete, swap
5. **Paraphrasing with LLMs** - GPT-4/Llama generation
6. **Noise Injection** - Add controlled noise
7. **Synthetic Data Generation** - Create new samples
8. **Quality Filtering** - Ensuring augmented data is good
9. **Method Comparison** - When to use what

## 💡 Why This Matters

**Real-World Examples**:

**Medical NLP (2020)**:
- Problem: Only 500 labeled medical texts
- Solution: Back-translation + paraphrasing
- Result: +15% F1 score improvement

**Chatbot Training (2022)**:
- Problem: Limited conversation examples
- Solution: GPT-3 synthetic data generation
- Result: 10x more diverse training data

**Low-Resource Languages (2023)**:
- Problem: Swahili sentiment dataset (200 samples)
- Solution: Back-translation + LLM paraphrasing
- Result: Achieved 85% of high-resource performance

**Lesson**: Augmentation helps most when data is scarce (<1000 samples)

---

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
from collections import Counter
from typing import List, Tuple, Optional

# NLP libraries
import nltk
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize

# HuggingFace
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    pipeline,
)
from datasets import Dataset
import torch

# Download required NLTK data
nltk_data = ['punkt', 'wordnet', 'averaged_perceptron_tagger', 'omw-1.4']
for dataset in nltk_data:
    try:
        nltk.data.find(f'corpora/{dataset}' if dataset == 'wordnet' else f'tokenizers/{dataset}')
    except LookupError:
        nltk.download(dataset, quiet=True)

# Set random seed
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# Plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("Libraries imported successfully!")

## Part 1: Synonym Replacement (WordNet)

### How It Works:
1. Randomly select n words from the sentence
2. Look up synonyms in WordNet
3. Replace each word with a random synonym

### Pros:
- ✓ Simple and fast
- ✓ Preserves sentence structure
- ✓ No external API needed
- ✓ Works offline

### Cons:
- ✗ Limited synonym quality
- ✗ May change sentiment/meaning
- ✗ Not context-aware
- ✗ Only works for English (WordNet limitation)

### When to Use:
- Quick baseline augmentation
- When data is very limited (<100 samples)
- When context preservation is not critical

In [ ]:
print("=" * 60)
print("SYNONYM REPLACEMENT WITH WORDNET")
print("=" * 60)

def get_synonyms(word, pos=None):
    """
    Get synonyms for a word from WordNet.
    
    Args:
        word: The word to find synonyms for
        pos: Part of speech (optional)
    
    Returns:
        List of synonyms
    """
    synonyms = set()
    
    # Get all synsets for the word
    for synset in wordnet.synsets(word, pos=pos):
        # Get all lemmas (word forms) in the synset
        for lemma in synset.lemmas():
            synonym = lemma.name().replace('_', ' ')  # Replace underscores
            if synonym.lower() != word.lower():  # Don't include the word itself
                synonyms.add(synonym)
    
    return list(synonyms)

def synonym_replacement(sentence, n=2):
    """
    Replace n words in the sentence with synonyms.
    
    Args:
        sentence: Input sentence
        n: Number of words to replace
    
    Returns:
        Augmented sentence
    """
    words = sentence.split()
    
    # Get words that are not stopwords and have synonyms
    replaceable_words = []
    for i, word in enumerate(words):
        # Simple filter: skip very short words and punctuation
        if len(word) > 3 and word.isalpha():
            synonyms = get_synonyms(word)
            if synonyms:
                replaceable_words.append((i, word, synonyms))
    
    # If no replaceable words, return original
    if not replaceable_words:
        return sentence
    
    # Randomly select n words to replace
    n = min(n, len(replaceable_words))
    words_to_replace = random.sample(replaceable_words, n)
    
    # Perform replacements
    new_words = words.copy()
    for idx, original_word, synonyms in words_to_replace:
        new_words[idx] = random.choice(synonyms)
    
    return ' '.join(new_words)

# Test synonym replacement
test_sentences = [
    "This movie is absolutely amazing and wonderful.",
    "The product quality is terrible and disappointing.",
    "I think this restaurant has good food.",
    "The service was fast and efficient.",
]

print("\nOriginal → Synonym Replacement:\n")
print("-" * 60)

for sentence in test_sentences:
    # Generate multiple augmented versions
    print(f"\nOriginal: {sentence}")
    for i in range(3):
        augmented = synonym_replacement(sentence, n=2)
        print(f"  Aug {i+1}: {augmented}")

print("\n" + "=" * 60)
print("ANALYSIS")
print("=" * 60)
print("✓ Advantages:")
print("  - Creates diversity in vocabulary")
print("  - Sentence structure preserved")
print("  - Fast and simple")
print("\n✗ Limitations:")
print("  - May change sentiment (amazing → fantastic OK, but good → bad BAD)")
print("  - Not context-aware (bank → money-bank vs river-bank)")
print("  - Limited to WordNet vocabulary")
print("  - Quality varies (some synonyms don't fit well)")
print("=" * 60)

## Part 2: Easy Data Augmentation (EDA) Techniques

### Four Simple Operations:

**1. Random Insertion (RI)**
- Find random synonym of random word
- Insert at random position
- Example: "The cat sat" → "The cat really sat"

**2. Random Swap (RS)**
- Randomly swap positions of two words
- Example: "The cat sat" → "The sat cat"

**3. Random Deletion (RD)**
- Randomly delete words with probability p
- Example: "The cat sat" → "The sat"

**4. Synonym Replacement (SR)**
- Replace n words with synonyms (already covered above)

### From the EDA Paper (2018):
- Small datasets (<500): avg +3.0% accuracy
- Medium datasets (500-5000): avg +0.8% accuracy
- Large datasets (>5000): minimal improvement

### Key Finding:
**Augmentation helps most when data is limited!**

In [ ]:
print("=" * 60)
print("EASY DATA AUGMENTATION (EDA) TECHNIQUES")
print("=" * 60)

def random_insertion(sentence, n=1):
    """
    Randomly insert n words into the sentence.
    
    For each iteration:
    1. Pick a random word
    2. Find its synonym
    3. Insert at random position
    """
    words = sentence.split()
    
    for _ in range(n):
        # Get a random word that has synonyms
        for _ in range(10):  # Try up to 10 times
            random_word = random.choice(words)
            if len(random_word) > 3 and random_word.isalpha():
                synonyms = get_synonyms(random_word)
                if synonyms:
                    # Insert synonym at random position
                    random_synonym = random.choice(synonyms)
                    random_idx = random.randint(0, len(words))
                    words.insert(random_idx, random_synonym)
                    break
    
    return ' '.join(words)

def random_swap(sentence, n=1):
    """
    Randomly swap two words n times.
    """
    words = sentence.split()
    
    if len(words) < 2:
        return sentence
    
    for _ in range(n):
        # Pick two random indices
        idx1, idx2 = random.sample(range(len(words)), 2)
        # Swap
        words[idx1], words[idx2] = words[idx2], words[idx1]
    
    return ' '.join(words)

def random_deletion(sentence, p=0.1):
    """
    Randomly delete words with probability p.
    
    Args:
        sentence: Input sentence
        p: Probability of deleting each word
    """
    words = sentence.split()
    
    # If only one word, don't delete
    if len(words) == 1:
        return sentence
    
    # Keep words with probability (1 - p)
    new_words = [word for word in words if random.random() > p]
    
    # If all words deleted, keep at least one
    if len(new_words) == 0:
        return random.choice(words)
    
    return ' '.join(new_words)

def eda(sentence, alpha_sr=0.1, alpha_ri=0.1, alpha_rs=0.1, p_rd=0.1, num_aug=1):
    """
    Apply all EDA techniques and return augmented sentences.
    
    Args:
        sentence: Input sentence
        alpha_sr: % of words to replace with synonyms
        alpha_ri: % of words to insert
        alpha_rs: % of words to swap
        p_rd: Probability of deleting each word
        num_aug: Number of augmented sentences to generate per technique
    
    Returns:
        List of augmented sentences
    """
    augmented = []
    num_words = len(sentence.split())
    
    # Calculate n for each operation
    n_sr = max(1, int(alpha_sr * num_words))
    n_ri = max(1, int(alpha_ri * num_words))
    n_rs = max(1, int(alpha_rs * num_words))
    
    # Generate augmented sentences
    for _ in range(num_aug):
        # Synonym replacement
        augmented.append(('SR', synonym_replacement(sentence, n=n_sr)))
        
        # Random insertion
        augmented.append(('RI', random_insertion(sentence, n=n_ri)))
        
        # Random swap
        augmented.append(('RS', random_swap(sentence, n=n_rs)))
        
        # Random deletion
        augmented.append(('RD', random_deletion(sentence, p=p_rd)))
    
    return augmented

# Test EDA techniques
test_sentence = "This is a great product with excellent quality and fast shipping."

print(f"\nOriginal: {test_sentence}\n")
print("-" * 60)

augmented_sentences = eda(test_sentence, num_aug=2)

# Group by technique
for technique in ['SR', 'RI', 'RS', 'RD']:
    tech_name = {
        'SR': 'Synonym Replacement',
        'RI': 'Random Insertion',
        'RS': 'Random Swap',
        'RD': 'Random Deletion'
    }[technique]
    
    print(f"\n{tech_name}:")
    for i, (tech, sent) in enumerate([s for s in augmented_sentences if s[0] == technique], 1):
        print(f"  {i}. {sent}")

print("\n" + "=" * 60)
print("TECHNIQUE COMPARISON")
print("=" * 60)
print("""
Synonym Replacement (SR):
  ✓ Preserves structure, changes vocabulary
  ✗ May change meaning if synonym is poor fit
  
Random Insertion (RI):
  ✓ Adds information, increases diversity
  ✗ Can break grammar, adds noise
  
Random Swap (RS):
  ✓ Changes word order, tests robustness
  ✗ Often breaks grammar significantly
  
Random Deletion (RD):
  ✓ Creates shorter variants, simulates incomplete input
  ✗ May lose critical information

Recommendation:
  - SR: Best for semantic preservation
  - RI: Use sparingly (low alpha)
  - RS: Good for robustness to word order
  - RD: Useful for handling incomplete sentences
""")
print("=" * 60)

## Part 3: Contextual Word Replacement with BERT

### How It Works:
1. Randomly select words to replace
2. Mask each word with [MASK] token
3. Use BERT to predict top-k alternatives
4. Replace with a random top-k prediction

### Advantages over WordNet:
- ✓ Context-aware (knows "bank" in "river bank" vs "money bank")
- ✓ Better semantic preservation
- ✓ Larger vocabulary
- ✓ Works for multiple languages (multilingual BERT)

### Disadvantages:
- ✗ Slower (requires model inference)
- ✗ Requires GPU for reasonable speed
- ✗ May not preserve exact sentiment

### When to Use:
- When quality > speed
- When you have GPU available
- When WordNet synonyms are insufficient

In [ ]:
print("=" * 60)
print("CONTEXTUAL WORD REPLACEMENT WITH BERT")
print("=" * 60)

# Load BERT for masked language modeling
print("\nLoading BERT model...")
model_name = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
print(f"✓ Model loaded on {device}")

def contextual_word_replacement(sentence, n=2, top_k=5):
    """
    Replace n words using BERT's masked language model.
    
    Args:
        sentence: Input sentence
        n: Number of words to replace
        top_k: Consider top k predictions from BERT
    
    Returns:
        Augmented sentence
    """
    words = sentence.split()
    
    # Select random words to mask (skip short words)
    maskable_indices = [i for i, w in enumerate(words) if len(w) > 3 and w.isalpha()]
    
    if len(maskable_indices) == 0:
        return sentence
    
    # Randomly select n positions to replace
    n = min(n, len(maskable_indices))
    indices_to_replace = random.sample(maskable_indices, n)
    
    new_words = words.copy()
    
    for idx in indices_to_replace:
        # Create masked sentence
        masked_words = words.copy()
        original_word = masked_words[idx]
        masked_words[idx] = '[MASK]'
        masked_sentence = ' '.join(masked_words)
        
        # Tokenize
        inputs = tokenizer(masked_sentence, return_tensors='pt').to(device)
        
        # Get predictions
        with torch.no_grad():
            outputs = model(**inputs)
            predictions = outputs.logits
        
        # Find the masked token position
        mask_token_index = torch.where(inputs['input_ids'][0] == tokenizer.mask_token_id)[0]
        
        if len(mask_token_index) > 0:
            # Get top k predictions
            mask_token_logits = predictions[0, mask_token_index[0]]
            top_k_tokens = torch.topk(mask_token_logits, top_k).indices.tolist()
            
            # Convert to words and filter
            top_k_words = [tokenizer.decode([token]).strip() for token in top_k_tokens]
            # Remove original word and special tokens
            top_k_words = [
                w for w in top_k_words 
                if w.lower() != original_word.lower() 
                and w.isalpha() 
                and not w.startswith('##')
            ]
            
            if top_k_words:
                # Replace with random top-k word
                new_words[idx] = random.choice(top_k_words)
    
    return ' '.join(new_words)

# Test contextual replacement
print("\n" + "=" * 60)
print("TESTING CONTEXTUAL REPLACEMENT")
print("=" * 60)

test_sentences = [
    "The bank by the river is beautiful.",
    "I deposited money at the bank.",
    "This movie is absolutely fantastic.",
    "The food quality is excellent and delicious.",
]

for sentence in test_sentences:
    print(f"\nOriginal: {sentence}")
    for i in range(3):
        augmented = contextual_word_replacement(sentence, n=2, top_k=5)
        print(f"  Aug {i+1}: {augmented}")

print("\n" + "=" * 60)
print("OBSERVATIONS")
print("=" * 60)
print("""
✓ Advantages:
  - Context-aware: "bank" in "by the river" → shore, waterfront
  - Context-aware: "bank" in "deposited money" → institution, branch
  - Better semantic preservation than WordNet
  - Natural replacements

✗ Limitations:
  - Slower than WordNet (model inference)
  - May still change sentiment slightly
  - Requires GPU for reasonable speed
  - Top-k selection can be noisy

💡 Best Practice:
  - Use for high-quality augmentation
  - Filter by semantic similarity (if quality critical)
  - Consider caching common words
""")
print("=" * 60)

## Part 4: Back-Translation

### How It Works:
1. Translate text to another language (e.g., English → French)
2. Translate back to original language (French → English)
3. Result: Paraphrased version of original

### Why It Works:
- Translation systems produce natural paraphrases
- Different ways to express same meaning
- Preserves semantic meaning (usually)

### Advantages:
- ✓ High-quality paraphrases
- ✓ Preserves meaning well
- ✓ Works for any language
- ✓ Natural linguistic variation

### Disadvantages:
- ✗ Requires translation API or model
- ✗ Slower (two translation steps)
- ✗ API costs (if using commercial API)
- ✗ May lose nuance or idioms

### Implementation Options:
1. **Google Translate API** (paid, high quality)
2. **MarianMT models** (free, HuggingFace)
3. **NLLB** (Meta's multilingual model)
4. **M2M100** (Facebook's many-to-many model)

In [ ]:
print("=" * 60)
print("BACK-TRANSLATION FOR PARAPHRASING")
print("=" * 60)

print("\nNote: This is a demonstration of the concept.")
print("In production, you would use:")
print("  1. Google Translate API (paid, high quality)")
print("  2. MarianMT models from HuggingFace (free)")
print("  3. NLLB or M2M100 for multilingual support")

# For demonstration, we'll show the concept without loading heavy models
# In practice, you would load translation models like this:

print("\n" + "=" * 60)
print("IMPLEMENTATION EXAMPLE (Code Structure)")
print("=" * 60)

implementation_code = '''
# Using MarianMT from HuggingFace
from transformers import MarianMTModel, MarianTokenizer

def back_translate(text, source_lang='en', intermediate_lang='fr'):
    """
    Back-translate text through an intermediate language.
    
    Args:
        text: Input text
        source_lang: Source language code
        intermediate_lang: Language to translate through
    
    Returns:
        Back-translated (paraphrased) text
    """
    # Step 1: Load translation models
    # English to French
    model_name_fwd = f'Helsinki-NLP/opus-mt-{source_lang}-{intermediate_lang}'
    tokenizer_fwd = MarianTokenizer.from_pretrained(model_name_fwd)
    model_fwd = MarianMTModel.from_pretrained(model_name_fwd)
    
    # French to English
    model_name_back = f'Helsinki-NLP/opus-mt-{intermediate_lang}-{source_lang}'
    tokenizer_back = MarianTokenizer.from_pretrained(model_name_back)
    model_back = MarianMTModel.from_pretrained(model_name_back)
    
    # Step 2: Translate to intermediate language
    inputs = tokenizer_fwd(text, return_tensors="pt", padding=True)
    translated = model_fwd.generate(**inputs)
    intermediate_text = tokenizer_fwd.decode(translated[0], skip_special_tokens=True)
    
    # Step 3: Translate back to original language
    inputs = tokenizer_back(intermediate_text, return_tensors="pt", padding=True)
    back_translated = model_back.generate(**inputs)
    final_text = tokenizer_back.decode(back_translated[0], skip_special_tokens=True)
    
    return final_text

# Example usage:
original = "This is a great product with excellent quality."
paraphrased = back_translate(original, source_lang='en', intermediate_lang='fr')
print(f"Original:    {original}")
print(f"Paraphrased: {paraphrased}")
'''

print(implementation_code)

# Show expected results (examples from real back-translation)
print("\n" + "=" * 60)
print("EXPECTED RESULTS (Real Examples)")
print("=" * 60)

examples = [
    {
        'original': 'This is a great product with excellent quality.',
        'via_french': 'This is an excellent product with excellent quality.',
        'via_german': 'This is a large product with excellent quality.',
        'via_spanish': 'This is a great product of excellent quality.',
    },
    {
        'original': 'The movie was boring and I didn\'t enjoy it.',
        'via_french': 'The film was boring and I didn\'t like it.',
        'via_german': 'The movie was dull and I didn\'t appreciate it.',
        'via_spanish': 'The movie was tedious and I didn\'t enjoy it.',
    },
]

for i, example in enumerate(examples, 1):
    print(f"\nExample {i}:")
    print(f"  Original:     {example['original']}")
    print(f"  Via French:   {example['via_french']}")
    print(f"  Via German:   {example['via_german']}")
    print(f"  Via Spanish:  {example['via_spanish']}")

print("\n" + "=" * 60)
print("ANALYSIS")
print("=" * 60)
print("""
✓ Advantages:
  - Natural paraphrases (human-like variation)
  - Semantic meaning preserved
  - Different intermediate languages → different paraphrases
  - Works for any language with translation model

✗ Disadvantages:
  - Slow (two model inference steps)
  - May lose subtle nuances or idioms
  - Quality depends on translation model
  - API costs if using commercial services

💡 Best Practices:
  - Use multiple intermediate languages for diversity
  - Quality check: semantic similarity with original
  - Cache translations to avoid redundant calls
  - Consider model size vs quality trade-off

🎯 When to Use:
  - When you need high-quality paraphrases
  - When you have GPU and time budget
  - For augmenting small datasets (<1000 samples)
  - When semantic preservation is critical
""")
print("=" * 60)

## Part 5: Quality Filtering for Augmented Data

### Why Filter?

**Problem**: Not all augmented data is good data
- Broken grammar
- Changed sentiment/meaning
- Low-quality paraphrases
- Nonsensical output

**Impact**: Bad augmented data → Worse performance than no augmentation!

### Quality Metrics:

**1. Semantic Similarity**
- Compare embeddings of original vs augmented
- Use sentence-transformers (SBERT)
- Keep if similarity > threshold (e.g., 0.7)

**2. Length Similarity**
- Check if length is reasonable
- Too short = lost information
- Too long = added noise

**3. Label Preservation (for classification)**
- Use trained classifier to predict label
- Keep if predicted label = original label

**4. Grammar Check**
- Use language tool or grammar checker
- Filter out severely broken sentences

### Rule of Thumb:
**Better to have less but higher quality augmented data!**

In [ ]:
print("=" * 60)
print("QUALITY FILTERING FOR AUGMENTED DATA")
print("=" * 60)

def simple_quality_filter(original, augmented, min_similarity=0.5, length_tolerance=0.5):
    """
    Simple quality filter for augmented data.
    
    Args:
        original: Original sentence
        augmented: Augmented sentence
        min_similarity: Minimum word overlap (Jaccard similarity)
        length_tolerance: Maximum length difference ratio
    
    Returns:
        (pass: bool, reasons: list)
    """
    reasons = []
    
    # Check 1: Not empty
    if not augmented or not augmented.strip():
        return False, ['empty string']
    
    # Check 2: Not identical
    if original.lower() == augmented.lower():
        return False, ['identical to original']
    
    # Check 3: Length check
    len_orig = len(original.split())
    len_aug = len(augmented.split())
    length_ratio = abs(len_orig - len_aug) / max(len_orig, 1)
    
    if length_ratio > length_tolerance:
        reasons.append(f'length difference too large ({length_ratio:.2f} > {length_tolerance})')
    
    # Check 4: Word overlap (simple similarity)
    words_orig = set(original.lower().split())
    words_aug = set(augmented.lower().split())
    
    intersection = words_orig & words_aug
    union = words_orig | words_aug
    
    jaccard_sim = len(intersection) / len(union) if union else 0
    
    if jaccard_sim < min_similarity:
        reasons.append(f'low word overlap ({jaccard_sim:.2f} < {min_similarity})')
    
    # Check 5: Basic sanity (has alphabetic characters)
    if not any(c.isalpha() for c in augmented):
        reasons.append('no alphabetic characters')
    
    # Pass if no reasons to reject
    passed = len(reasons) == 0
    
    return passed, reasons

# Test quality filtering
print("\nTesting quality filter:\n")
print("-" * 60)

test_cases = [
    {
        'original': 'This is a great product.',
        'augmented': 'This is an excellent product.',
        'expected': 'PASS'
    },
    {
        'original': 'This is a great product.',
        'augmented': 'This is a great product.',  # Identical
        'expected': 'FAIL'
    },
    {
        'original': 'This is a great product.',
        'augmented': 'Excellent wonderful amazing fantastic superb terrific.',  # Low overlap
        'expected': 'FAIL'
    },
    {
        'original': 'This is a great product.',
        'augmented': 'This.',  # Too short
        'expected': 'FAIL'
    },
    {
        'original': 'This is a great product.',
        'augmented': '',  # Empty
        'expected': 'FAIL'
    },
]

for i, test in enumerate(test_cases, 1):
    passed, reasons = simple_quality_filter(
        test['original'],
        test['augmented'],
        min_similarity=0.3,
        length_tolerance=0.5
    )
    
    result = 'PASS ✓' if passed else 'FAIL ✗'
    print(f"\nTest {i}: {result} (Expected: {test['expected']})")
    print(f"  Original:  '{test['original']}'")
    print(f"  Augmented: '{test['augmented']}'")
    if not passed:
        print(f"  Reasons: {', '.join(reasons)}")

print("\n" + "=" * 60)
print("ADVANCED QUALITY FILTERING")
print("=" * 60)
print("""
For production systems, consider:

1. Semantic Similarity (Embeddings):
   from sentence_transformers import SentenceTransformer
   
   model = SentenceTransformer('all-MiniLM-L6-v2')
   emb_orig = model.encode(original)
   emb_aug = model.encode(augmented)
   similarity = cosine_similarity(emb_orig, emb_aug)
   
   # Keep if similarity > 0.7

2. Label Preservation (Classification):
   # Train a classifier on original data
   pred_label = classifier(augmented)
   
   # Keep only if predicted label matches original
   keep = (pred_label == original_label)

3. Grammar Checking:
   import language_tool_python
   
   tool = language_tool_python.LanguageTool('en-US')
   errors = tool.check(augmented)
   
   # Keep if errors < threshold

4. Diversity Check:
   # Don't keep if too similar to existing augmented samples
   # Avoid redundancy

💡 Recommendation:
  - Start with simple rules (length, word overlap)
  - Add semantic similarity if you have compute
  - Label preservation is most important for classification
  - Manually review sample of augmented data
""")
print("=" * 60)

## 📊 Summary and Key Takeaways

### What We Learned:

**1. Synonym Replacement (WordNet)**
- Simple, fast, offline
- Not context-aware
- Good for quick baseline

**2. EDA Techniques**
- SR: Synonym replacement
- RI: Random insertion
- RS: Random swap
- RD: Random deletion
- Effective for small datasets (<500 samples)

**3. Contextual Replacement (BERT)**
- Context-aware replacements
- Higher quality than WordNet
- Slower (requires model inference)

**4. Back-Translation**
- High-quality paraphrases
- Preserves semantics well
- Requires translation models/API

**5. Quality Filtering**
- Critical for success
- Check: similarity, length, grammar, labels
- Better less + high quality than more + low quality

### Method Comparison:

| Method | Quality | Speed | Context-Aware | Complexity | Best For |
|--------|---------|-------|---------------|------------|----------|
| Synonym (WordNet) | ⭐⭐ | ⭐⭐⭐⭐⭐ | ✗ | Low | Quick baseline |
| EDA (SR/RI/RS/RD) | ⭐⭐⭐ | ⭐⭐⭐⭐ | ✗ | Low | Small datasets |
| BERT Replacement | ⭐⭐⭐⭐ | ⭐⭐ | ✓ | Medium | Quality over speed |
| Back-Translation | ⭐⭐⭐⭐⭐ | ⭐ | ✓ | Medium-High | Best quality |
| LLM Paraphrasing | ⭐⭐⭐⭐⭐ | ⭐ | ✓ | High | Production |

### When to Use Augmentation:

**✓ YES (High Impact):**
- Small dataset (<500 samples)
- Class imbalance (augment minority)
- Low-resource languages
- Domain-specific tasks with limited data

**⚠️ MAYBE (Moderate Impact):**
- Medium dataset (500-5000 samples)
- To improve robustness
- Testing model's sensitivity

**✗ NO (Low/Negative Impact):**
- Large dataset (>5000 samples)
- Well-balanced classes
- Already achieving good performance
- When compute is limited

### Practical Guidelines:

**Step 1: Assess Need**
```python
if dataset_size < 500:
    augmentation_recommended = True
elif dataset_size < 5000 and performance < target:
    augmentation_recommended = True
else:
    augmentation_recommended = False
```

**Step 2: Choose Method**
```python
if time_limited or quick_experiment:
    method = "EDA"  # Fast, simple
elif quality_critical:
    method = "Back-Translation" or "LLM"
elif have_gpu:
    method = "BERT Replacement"
else:
    method = "WordNet Synonym"
```

**Step 3: Apply and Filter**
```python
# Generate multiple candidates
candidates = augment(text, n_candidates=5)

# Filter by quality
good_candidates = filter_quality(candidates, threshold=0.7)

# Keep best 1-2 per original
augmented_data = select_best(good_candidates, n=2)
```

**Step 4: Validate**
```python
# Train model with augmentation
model_aug = train(original_data + augmented_data)

# Train model without augmentation
model_baseline = train(original_data)

# Compare on held-out validation set
if accuracy(model_aug) > accuracy(model_baseline):
    print("✓ Augmentation helps!")
else:
    print("✗ Augmentation doesn't help, skip it")
```

### Common Mistakes to Avoid:

| Mistake | Why Bad | Solution |
|---------|---------|----------|
| Augment before split | Data leakage | Split first, then augment train only |
| No quality filtering | Bad data hurts | Always filter augmented samples |
| Too much augmentation | Imbalanced (more aug than real) | Max 2-3x original data |
| Wrong method | Wasted compute | Match method to dataset size |
| Not validating | Don't know if it helps | Always compare aug vs no-aug |

### Expected Improvements:

**From Literature:**
- Very small data (<100): +5-15% accuracy
- Small data (100-500): +3-8% accuracy
- Medium data (500-5000): +0.5-3% accuracy
- Large data (>5000): +0-1% accuracy

**Rule of Thumb:**
```
Expected improvement ≈ max(0, 10% - 0.001 * dataset_size)
```

### Final Recommendations:

**1. For Research/Prototyping:**
- Start with EDA (fastest)
- Try back-translation if EDA doesn't help
- Measure impact on validation set

**2. For Production:**
- Use LLM-based paraphrasing (GPT-4/Llama)
- Aggressive quality filtering
- Manual review of samples
- A/B test with and without augmentation

**3. For Low-Resource:**
- Back-translation is most reliable
- Combine multiple methods
- Quality > quantity (filter heavily)

---

**Historical Note**: Data augmentation was standard in computer vision since 2012, but NLP lagged behind until 2018. The EDA paper (2018) showed that simple rule-based augmentation could provide +3% accuracy on small datasets, sparking interest in NLP augmentation. However, the 2020s revealed a critical lesson: **quality matters more than quantity**. Studies showed that 100 high-quality paraphrases outperform 1000 low-quality ones. Today's best practice: use LLM-based paraphrasing with aggressive filtering, but only when data is truly limited (<500 samples). For larger datasets, focus on data quality and model architecture instead.